# Visualisation — Metadata ResStock (sous-ensemble national)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Racine du projet — 2 niveaux au-dessus de ce notebook
# notebooks/03_visualisation/ -> notebooks/ -> FlexiMax/
ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

print('Racine projet :', ROOT)

## 1. Chargement du sous-ensemble

In [ ]:
# Colonnes qu'on veut garder (on ne charge pas les 771 colonnes)
colonnes = [
    'bldg_id',
    'in.state',
    'in.ashrae_iecc_climate_zone_2004',
    'in.geometry_floor_area',
    'in.hvac_heating_type',
    'in.hvac_cooling_type',
    'in.vintage',
    'out.electricity.total.energy_consumption..kwh',
    'out.electricity.cooling.energy_consumption..kwh',
    'out.electricity.heating.energy_consumption..kwh'
]

# read_parquet() lit le fichier parquet — on charge seulement les colonnes choisies
df_all = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=colonnes)

# sample() tire un sous-ensemble aleatoire — ici 5000 batiments sur ~550 000
# random_state=42 garantit qu'on obtient toujours le meme sous-ensemble
df = df_all.sample(n=5000, random_state=42).reset_index(drop=True)

print('Taille du sous-ensemble :', df.shape)
df.head()

## 2. Distribution par zone climatique ASHRAE

In [ ]:
# value_counts() compte le nombre de batiments par zone climatique
zones = df['in.ashrae_iecc_climate_zone_2004'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(zones.index, zones.values, color='steelblue')
ax.set_title('Nombre de batiments par zone climatique ASHRAE')
ax.set_xlabel('Zone climatique')
ax.set_ylabel('Nombre de batiments')
plt.tight_layout()
plt.show()

## 3. Distribution des surfaces (floor area)

In [ ]:
# in.geometry_floor_area est une categorie string comme '1000-1499'
ordre = ['0-499', '500-749', '750-999', '1000-1499', '1500-1999',
         '2000-2499', '2500-2999', '3000-3999', '4000+']

surfaces = df['in.geometry_floor_area'].value_counts()
surfaces = surfaces.reindex([o for o in ordre if o in surfaces.index])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(surfaces.index, surfaces.values, color='coral')
ax.set_title('Distribution des surfaces des batiments (ft2)')
ax.set_xlabel('Surface (ft2)')
ax.set_ylabel('Nombre de batiments')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Types de chauffage HVAC

In [ ]:
# On compte les types de chauffage et on garde les 10 plus frequents
chauffage = df['in.hvac_heating_type'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 4))
# barh() trace des barres horizontales — plus lisible quand les labels sont longs
ax.barh(chauffage.index, chauffage.values, color='mediumseagreen')
ax.set_title('Types de chauffage les plus frequents (top 10)')
ax.set_xlabel('Nombre de batiments')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Consommation electrique totale par zone climatique

In [ ]:
zones_list = sorted(df['in.ashrae_iecc_climate_zone_2004'].dropna().unique())

# Pour chaque zone, on recupere les valeurs de consommation
# dropna() supprime les valeurs manquantes avant la boxplot
data_boxplot = [
    df[df['in.ashrae_iecc_climate_zone_2004'] == z]['out.electricity.total.energy_consumption..kwh'].dropna().values
    for z in zones_list
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(data_boxplot, labels=zones_list, patch_artist=True)
ax.set_title('Consommation electrique totale par zone climatique ASHRAE (kWh/an)')
ax.set_xlabel('Zone climatique')
ax.set_ylabel('Consommation (kWh/an)')
plt.tight_layout()
plt.show()